# Unit 1：ICA 的数学直觉 —— 鸡尾酒会问题

## 目标
- 理解「盲源分离」概念
- 用代码模拟鸡尾酒会问题
- 直观感受 ICA 如何从混合信号中恢复源信号

## 核心问题
> 房间里三个人同时说话，两个麦克风录下混合信号。
> **只凭混合录音，能否还原每个人的原始语音？**


### 1.1 生成模拟"源信号"

我们用三种不同的信号模拟"说话者"：
- 正弦波（100 Hz，光滑振荡）
- 正弦波（300 Hz，不同相位）
- 方波（500 Hz，陡峭切换）


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 参数
fs = 2000           # 采样率 2000 Hz
t = np.arange(0, 1, 1/fs)  # 1 秒
n_samples = len(t)

# 三个「说话者」——三种不同的源信号
s1 = np.sin(2 * np.pi * 100 * t)              # 100 Hz 正弦波
s2 = np.sin(2 * np.pi * 300 * t + 1.5)        # 300 Hz 正弦波（相位偏移）
s3 = np.sign(np.sin(2 * np.pi * 500 * t))     # 500 Hz 方波

S = np.vstack([s1, s2, s3])  # shape: (3, n_samples)

# 绘图
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
labels = ['源信号 1 (100 Hz 正弦)', '源信号 2 (300 Hz 正弦)', '源信号 3 (500 Hz 方波)']
for ax, s, label in zip(axes, S, labels):
    ax.plot(t[:200], s[:200], linewidth=0.8)
    ax.set_ylabel(label, fontsize=9)
    ax.set_xlim(0, 0.1)
axes[-1].set_xlabel('时间 (s)')
fig.suptitle('源信号（三个独立信号源）', fontsize=14)
plt.tight_layout()
plt.show()


### 1.2 混合信号 —— 模拟麦克风录音

每个麦克风收到的是三个源信号的**线性加权和**：

$$X = A \cdot S$$

其中 $A$ 是混合矩阵（未知），$X$ 是观测信号（已知）。


In [ ]:
# 随机混合矩阵（模拟麦克风位置不同导致的权重差异）
np.random.seed(42)
A = np.random.randn(3, 3)
X = A @ S  # (3, 3) @ (3, n_samples) = (3, n_samples)

print("混合矩阵 A（在真实问题中未知！）:")
print(np.array2string(A, precision=3, suppress_small=True))

# 可视化混合信号
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
for ax, x, label in zip(axes, X, ['麦克风 1', '麦克风 2', '麦克风 3']):
    ax.plot(t[:200], x[:200], linewidth=0.8, color='crimson')
    ax.set_ylabel(label)
    ax.set_xlim(0, 0.1)
axes[-1].set_xlabel('时间 (s)')
fig.suptitle('观测信号（麦克风录到的混合信号）—— 已看不出原始形态', fontsize=14)
plt.tight_layout()
plt.show()


### 1.3 ICA 的核心假设

| 假设 | 含义 |
|------|------|
| **统计独立** | 源信号彼此独立（说话者互不影响） |
| **非高斯性** | 源信号最多一个高斯分布（中心极限定理保证混合信号更高斯） |
| **线性混合** | $X = AS$，瞬时线性混合 |
| **源数 ≤ 观测数** | 麦克风数量 ≥ 说话者数量 |

### 1.4 从混合信号恢复源信号


In [ ]:
from sklearn.decomposition import FastICA

# FastICA 尝试从 X 中恢复独立成分
ica = FastICA(n_components=3, random_state=42, max_iter=2000)
S_hat = ica.fit_transform(X.T).T  # 恢复的源信号

# 可视化对比：原始 vs 恢复
fig, axes = plt.subplots(3, 2, figsize=(14, 7), sharex=True, sharey='row')
for i in range(3):
    axes[i, 0].plot(t[:200], S[i, :200], linewidth=0.8)
    axes[i, 0].set_ylabel(f'源 {i+1}')
    axes[i, 1].plot(t[:200], S_hat[i, :200], linewidth=0.8, color='green')
    axes[i, 1].set_ylabel(f'恢复 {i+1}')
axes[0, 0].set_title('原始源信号')
axes[0, 1].set_title('ICA 恢复的信号')
axes[-1, 0].set_xlabel('时间 (s)')
axes[-1, 1].set_xlabel('时间 (s)')
fig.suptitle('ICA 盲源分离效果对比', fontsize=14)
plt.tight_layout()
plt.show()

print("✅ 注意：ICA 恢复的信号可能有符号翻转和幅值缩放，但波形形态被完美保留！")


### 1.5 相关性验证

用 Pearson 相关系数量化恢复效果（绝对值越大越好）。


In [ ]:
from scipy.stats import pearsonr

print("源信号 vs 恢复信号 的相关性矩阵:")
print("        恢复1     恢复2     恢复3")
for i in range(3):
    corrs = [pearsonr(S[i], S_hat[j])[0] for j in range(3)]
    print(f"源{i+1}  {corrs[0]:>8.3f}  {corrs[1]:>8.3f}  {corrs[2]:>8.3f}")

# 每行最大值应该接近 1 或 -1
print("\n✅ 每行都有一个恢复信号与之高度相关 → ICA 成功分离！")
print("   注意符号（±）无所谓，波形形态才是关键。")


### 1.6 ICA 的两种固有模糊性

ICA 存在两个不可消除的不确定性（但在 EEG 中都不是问题）：

1. **幅值模糊**：恢复信号的幅值可能被缩放（$s$ vs $k \cdot s$）
   → EEG 中我们只关心波形形态，不要求绝对电压

2. **顺序模糊**：成分的排列顺序是随机的
   → 我们通过地形图/时间序列来「识别」成分，不依赖顺序

### 💡 关键理解

1. **ICA 不需要知道混合矩阵 A** —— 这就是「盲」源分离
2. **ICA 利用的是统计独立性** —— 不是频率差异、不是空间位置
3. **恢复的信号形态正确，但幅值和顺序可能变化**

### 🤔 思考题

- 如果两个源信号都是纯正弦波，ICA 还能分离吗？试试看。
- 如果麦克风数 < 说话者数（欠定问题），会发生什么？

→ 进入 **Unit 2：ICA 算法原理**
